# Day 058 — Solution: Streaming Chat API

In [ ]:
_STREAMING_API_SRC = '"""streaming_api.py — Day 058: streaming chat API with SSE and WebSocket.\n\nRun:  uvicorn streaming_api:app --reload\nDocs: http://localhost:8000/docs\n"""\nimport json\nimport os\nfrom datetime import datetime\n\nimport ollama\nfrom fastapi import FastAPI, WebSocket, WebSocketDisconnect\nfrom fastapi.middleware.cors import CORSMiddleware\nfrom fastapi.responses import StreamingResponse\nfrom pydantic import BaseModel, Field\n\nMODEL       = os.environ.get("MODEL", "llama3.2")\nAPP_VERSION = "1.0.0"\n\n\nclass ChatRequest(BaseModel):\n    prompt: str = Field(min_length=1)\n    system: str = ""\n\n\napp = FastAPI(title="Streaming Chat API", version=APP_VERSION)\napp.add_middleware(\n    CORSMiddleware,\n    allow_origins=["*"],\n    allow_credentials=False,\n    allow_methods=["*"],\n    allow_headers=["*"],\n)\n\n\n@app.get("/health")\ndef health():\n    return {\n        "status": "ok",\n        "timestamp": datetime.utcnow().isoformat() + "Z",\n        "version": APP_VERSION,\n    }\n\n\n@app.get("/stream/count")\ndef stream_count(n: int = 5):\n    """Demo SSE endpoint — streams n count events (no Ollama)."""\n    def generate():\n        for i in range(n):\n            yield "data: " + str(i) + "\\n\\n"\n        yield "data: [DONE]\\n\\n"\n    return StreamingResponse(generate(), media_type="text/event-stream")\n\n\n@app.post("/chat/stream")\ndef chat_stream(req: ChatRequest):\n    """Stream Ollama response tokens as SSE events."""\n    messages = []\n    if req.system:\n        messages.append({"role": "system", "content": req.system})\n    messages.append({"role": "user", "content": req.prompt})\n\n    def generate():\n        chunks = ollama.chat(model=MODEL, messages=messages, stream=True)\n        for chunk in chunks:\n            token = chunk["message"]["content"]\n            if token:\n                payload = json.dumps({"token": token})\n                yield "data: " + payload + "\\n\\n"\n        yield "data: [DONE]\\n\\n"\n\n    return StreamingResponse(generate(), media_type="text/event-stream")\n\n\n@app.websocket("/ws")\nasync def websocket_chat(ws: WebSocket):\n    """WebSocket endpoint — receives prompts, streams response token by token."""\n    await ws.accept()\n    try:\n        while True:\n            prompt = await ws.receive_text()\n            chunks = ollama.chat(\n                model=MODEL,\n                messages=[{"role": "user", "content": prompt}],\n                stream=True,\n            )\n            for chunk in chunks:\n                token = chunk["message"]["content"]\n                if token:\n                    await ws.send_text(token)\n            await ws.send_text("[DONE]")\n    except WebSocketDisconnect:\n        pass\n\n\nif __name__ == "__main__":\n    import uvicorn\n    PORT = int(os.environ.get("PORT", "8000"))\n    uvicorn.run(app, host="0.0.0.0", port=PORT)\n'
from pathlib import Path
Path('streaming_api.py').write_text(_STREAMING_API_SRC)
print('streaming_api.py written.')

In [ ]:
import json
from datetime import datetime
from fastapi import FastAPI, WebSocket, WebSocketDisconnect
from fastapi.responses import StreamingResponse
from pydantic import BaseModel, Field
from starlette.testclient import TestClient

# ── inline test app (no Ollama required) ──────────────────────────────────────
class _ChatReq(BaseModel):
    prompt: str = Field(min_length=1)

test_app = FastAPI()

@test_app.get("/health")
def _health():
    return {"status": "ok",
            "timestamp": datetime.utcnow().isoformat() + "Z",
            "version": "1.0.0"}

@test_app.get("/stream/count")
def _stream_count(n: int = 5):
    def gen():
        for i in range(n):
            yield "data: " + str(i) + "\n\n"
        yield "data: [DONE]\n\n"
    return StreamingResponse(gen(), media_type="text/event-stream")

@test_app.post("/chat/stream")
def _chat_stream(req: _ChatReq):
    def gen():
        for token in ["Day", " 058", " complete!"]:
            payload = json.dumps({"token": token})
            yield "data: " + payload + "\n\n"
        yield "data: [DONE]\n\n"
    return StreamingResponse(gen(), media_type="text/event-stream")

@test_app.websocket("/ws")
async def _ws(ws: WebSocket):
    await ws.accept()
    try:
        while True:
            data = await ws.receive_text()
            await ws.send_text(f"Echo: {data}")
    except WebSocketDisconnect:
        pass

client = TestClient(test_app, raise_server_exceptions=False)

# /health
r = client.get("/health")
assert r.status_code == 200 and r.json()["status"] == "ok"
print("\u2705 /health works")

# /stream/count
r2 = client.get("/stream/count?n=3")
assert r2.status_code == 200
assert "text/event-stream" in r2.headers["content-type"]
assert "data: 0" in r2.text and "[DONE]" in r2.text
print("\u2705 GET /stream/count streams SSE")

# /chat/stream
r3 = client.post("/chat/stream", json={"prompt": "hello"})
assert r3.status_code == 200
assert "text/event-stream" in r3.headers["content-type"]
assert '"token"' in r3.text and "[DONE]" in r3.text
print("\u2705 POST /chat/stream streams SSE tokens")

# empty prompt → 422
r4 = client.post("/chat/stream", json={"prompt": ""})
assert r4.status_code == 422
print("\u2705 empty prompt \u2192 422")

# WebSocket
with client.websocket_connect("/ws") as ws:
    ws.send_text("ping")
    msg = ws.receive_text()
    assert msg == "Echo: ping", f"Got {msg!r}"
print("\u2705 WebSocket /ws echoes messages")

print("\nDay 058 \u2014 Streaming in Web Apps complete! \U0001f389")
